# Korn til Furuset — følgehefte, runde 005

## tl;dr
Leverbar mengde og behov er ukjent for begge alternativer. Åtte avlingsvarsler består. Finske 2018-rester er 6,90 og 17,09 tusen tonn. Én felles fuktfaktor alene er utilstrekkelig under avrundingskontrollen. Dette er intern analyse; ingen prøve eller menneskelig review er utført.

## Context & Methods
Én mottaker: Møllhausen Furuset. Q1: sterkt siktet bakerihvetemel med Regal-standard som referanse. Frist: 72 timer i det designede scenarioet i `case.json`.

### Key Assumptions
10/20/40 tonn er illustrative melmål, ikke mottakerbehov. 0,78 er en katalogreferanse fra februar 2021; 0,70/0,85 er sensitiviteter. Fysisk fuktbasis og faktisk utbytte er ukjent. Finland-kontrollen bruker konservativ ±0,05 tusen tonn i begge kilder; dette er ikke et konfidensintervall. Norsk hvete beholder C1110; ingen felles nordisk hvetetotal.

Kjør fra repository eller sett `C1_REPO_ROOT`. Private originaler kreves på stiene i manifestet; eventuelt sett `C1_INPUT_MANIFEST` til en ny, eksplisitt bundet manifestkopi. Koden henter ikke nettdata.

In [1]:
from pathlib import Path
import os, json, importlib.util
repo = Path(os.environ.get("C1_REPO_ROOT", Path.cwd())).resolve()
while not (repo / "scripts/analyze-beredskap-c1.py").is_file():
    if repo == repo.parent:
        raise FileNotFoundError("Set C1_REPO_ROOT to the repository")
    repo = repo.parent
package = repo / "docs/project/analysis/source-review-beredskap-2026-09-09/round-005"
spec = importlib.util.spec_from_file_location("c1", repo / "scripts/analyze-beredskap-c1.py")
c1 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(c1)

## Data
Frosne Eurostat-, SSB-, SCB-, Luke- og HST77-uttrekk og caseprofil. Ni inputfiler; alle hashkontrolleres. Eksakte koordinater følger beregningene. `source-bindings.json` dokumenterer råkilder og leseomfang. Kildeversjonene er historiske/ervervet 9. september 2026, ikke sanntidslager.

In [2]:
manifest_path = Path(os.environ.get("C1_INPUT_MANIFEST", package / "input-manifest.json"))
manifest = json.loads(manifest_path.read_text())
inputs = c1.checked_inputs(manifest)
result = c1.analyze(inputs)
print("Hashkontrollerte inputfiler:", len(inputs))
print("Unike avlingsvarsler:", len(result["yieldDiagnostics"]))

Hashkontrollerte inputfiler: 9
Unike avlingsvarsler: 8


## Results
### Betinget råvarekrav
Bare dimensjonering; ingen leveranse- eller behovsobservasjon.

In [3]:
print("Melmål t | Antatt utbytte | Kornkrav t")
for row in result["designArithmetic"]:
    print(f'{row["illustrative_flour_target_t"]:8.0f} | {row["assumed_extraction_fraction"]:.2f}           | {row["grain_required_t"]:.3f}')

Melmål t | Antatt utbytte | Kornkrav t
      10 | 0.70           | 14.286
      10 | 0.78           | 12.821
      10 | 0.85           | 11.765
      20 | 0.70           | 28.571
      20 | 0.78           | 25.641
      20 | 0.85           | 23.529
      40 | 0.70           | 57.143
      40 | 0.78           | 51.282
      40 | 0.85           | 47.059


### Åtte uløste avlingsvarsler og Finland
Residual: rapportert avling minus P/A, i kg/ha.

In [4]:
for row in result["yieldDiagnostics"]:
    print(row["country"], row["crop_code"], row["year"], round(row["residual_kg_ha"], 3), "kg/ha; uløst")
for row in result["finland2018"]:
    print(row["crop"], round(row["residual_kt"], 2), "tusen tonn; uløst")
print("Felles fuktfaktor forenlig med begge:", result["commonMoistureOnlyFactorCompatibleWithBoth"])

NO C1110 2015 -174.429 kg/ha; uløst
NO C1110 2016 20.249 kg/ha; uløst
NO C1110 2017 14.521 kg/ha; uløst
NO C1300 2017 12.674 kg/ha; uløst
SE C1100 2015 6.799 kg/ha; uløst
DK C1300 2015 -9.873 kg/ha; uløst
FI C1100 2020 6.177 kg/ha; uløst
FI C1300 2016 7.323 kg/ha; uløst
wheat 6.9 tusen tonn; uløst
barley 17.09 tusen tonn; uløst
Felles fuktfaktor forenlig med begge: False


### Ingen observerte leveransetall
Ukjent bevares som `None`/JSON `null`, aldri 0 tonn.

In [5]:
assert all(result["delivery"][k] is None for k in ["national_t", "nordic_t", "netNeed_t", "nordicAdditional_t"])
assert result == json.loads((package / "calculations.json").read_text())
print(result["delivery"])
print("Beregninger identiske med kontrollert leveransefil.")

{'national_t': None, 'nordic_t': None, 'netNeed_t': None, 'nordicAdditional_t': None, 'reason': 'No recipient-bound stock, quality, allocation, throughput or delivery observations'}
Beregninger identiske med kontrollert leveransefil.


## Takeaways
Caset og målefeltene er konkrete, men Q1-godkjenning, behov, lager, allokering, gjennomstrømning og kvitteringer mangler. Les `MAALEPROTOKOLL.md`, `DEFINISJONER.md` og `NESTE-SESJON.md`. Kjøring av følgeheftet gir ingen fysisk, menneskelig eller publiseringsmessig autoritet.